# Module 2.3 — Linear Regression: The Simplest Model

**Module 2: Data & ML Fundamentals | Week 1 | Lecture 3**

---

Last session we learned how data is structured — features and labels, train/test splits. Now we build our first model.

Linear regression is the "hello world" of machine learning. It's simple enough to fully understand, but powerful enough to solve real problems. More importantly, the concepts behind it — loss functions, fitting, gradient descent — carry forward to every model we'll build in this course.

## Glossary

A lot of vocabulary here, but most of it names things you've seen before in math class — just with ML labels on them.

- **Weight** — a number the model multiplies a feature by. In `y = wx + b`, `w` is the weight. (In algebra class this is the *slope*.)
- **Bias** — a number the model adds at the end. In `y = wx + b`, `b` is the bias. (In algebra class this is the *intercept*.)
- **Loss function** (also called *cost function* or *error function*) — measures how wrong a model's predictions are. Smaller loss = better predictions. Three names, same thing.
- **MSE** — mean squared error. The most common loss for regression. Takes the average of (prediction − actual)² across all samples.
- **RMSE** — root mean squared error. Square root of MSE. Reported because it's in the same units as the target — easier to interpret.
- **Gradient descent** — the algorithm that finds the best weights by following the slope of the loss function downhill. The workhorse of ML training.
- **Learning rate** — the size of each step gradient descent takes. Too small and learning is slow; too big and the algorithm overshoots.
- **Convex** — a loss function with a single bowl-shaped minimum (no traps). Linear regression's MSE is convex, which is why gradient descent always finds the best answer for it.

The terms **weight**, **bias**, **loss**, **gradient descent**, **learning rate** all reappear in Module 3 with neural networks — the names stay the same, the math gets bigger.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

---

## What Is Linear Regression?

Linear regression finds the best straight line through your data.

You have a bunch of data points. You want a line that gets as close to all of them as possible. That's it.

The line has the form:

$$\hat{y} = w \cdot x + b$$

- $x$ = the feature (input)
- $\hat{y}$ = the prediction ("y-hat")
- $w$ = the **weight** (slope of the line) — how much $y$ changes when $x$ changes by 1
- $b$ = the **bias** (y-intercept) — the prediction when $x$ is 0

The model "learns" by finding the values of $w$ and $b$ that make the best predictions.

In [ ]:
# How linear regression works — input to output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 2.8))
ax.set_xlim(0, 11)
ax.set_ylim(0, 3)
ax.axis("off")

# Boxes: Input features → Linear combination → Prediction
boxes = [
    (1.5, "Input\nFeatures (x)", "#4A90D9"),
    (5.5, "Multiply & Add\nw · x + b", "#D9534F"),
    (9.5, "Prediction\nŷ", "#5CB85C"),
]

for x, label, color in boxes:
    box = mpatches.FancyBboxPatch(
        (x - 1.2, 0.7), 2.4, 1.6, boxstyle="round,pad=0.15",
        facecolor=color, edgecolor="white", linewidth=2, alpha=0.85
    )
    ax.add_patch(box)
    ax.text(x, 1.5, label, ha="center", va="center",
            fontsize=11, fontweight="bold", color="white")

# Arrows
ax.annotate("", xy=(4.25, 1.5), xytext=(2.75, 1.5),
            arrowprops=dict(arrowstyle="-|>", color="#333", lw=2))
ax.annotate("", xy=(8.25, 1.5), xytext=(6.75, 1.5),
            arrowprops=dict(arrowstyle="-|>", color="#333", lw=2))

# Example annotation
ax.text(5.5, 0.25, "e.g.  5 hours studied  →  8 × 5 + 25  →  65 points",
        ha="center", fontsize=9.5, color="#555", fontstyle="italic")

fig.suptitle("Linear Regression: Input → Output", fontsize=13, fontweight="bold", y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Let's create a simple dataset to work with
np.random.seed(42)
study_hours = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)
exam_scores = study_hours * 8 + 25 + np.random.normal(0, 5, 10)  # ~8 points per hour + 25 base

plt.figure(figsize=(8, 5))
plt.scatter(study_hours, exam_scores, s=80, zorder=5)
plt.xlabel("Hours Studied")
plt.ylabel("Exam Score")
plt.title("Study Hours vs Exam Score")
plt.grid(True, alpha=0.3)
plt.show()

There's clearly a pattern. More study hours → higher scores. A line would fit this well.

But which line? There are infinitely many lines you could draw. We need a way to measure which line is **best**.

---

## Loss Functions: Measuring "How Wrong"

A **loss function** (also called cost function or error function) measures how far off your predictions are from the actual values. Lower loss = better model.

The most common loss for regression is **Mean Squared Error (MSE)**:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In plain English: for each data point, calculate the error (actual minus predicted), square it, then average all the squared errors.

Why square the errors? Two reasons:
1. It makes all errors positive (an error of -5 is just as bad as +5)
2. It punishes big errors more than small ones (an error of 10 is much worse than two errors of 5)

In [ ]:
# Let's visualize what loss means with a BAD line and a GOOD line

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bad line: y = 3x + 50 (too flat)
bad_pred = 3 * study_hours + 50
bad_mse = np.mean((exam_scores - bad_pred) ** 2)

axes[0].scatter(study_hours, exam_scores, s=80, zorder=5)
axes[0].plot(study_hours, bad_pred, 'r-', linewidth=2)
for i in range(len(study_hours)):
    axes[0].plot([study_hours[i], study_hours[i]],
                 [exam_scores[i], bad_pred[i]], 'r--', alpha=0.5)
axes[0].set_title(f"Bad line — MSE = {bad_mse:.1f}")
axes[0].set_xlabel("Hours Studied")
axes[0].set_ylabel("Exam Score")
axes[0].grid(True, alpha=0.3)

# Good line: use sklearn to find the best one
model = LinearRegression()
model.fit(study_hours.reshape(-1, 1), exam_scores)
good_pred = model.predict(study_hours.reshape(-1, 1))
good_mse = np.mean((exam_scores - good_pred) ** 2)

axes[1].scatter(study_hours, exam_scores, s=80, zorder=5)
axes[1].plot(study_hours, good_pred, 'g-', linewidth=2)
for i in range(len(study_hours)):
    axes[1].plot([study_hours[i], study_hours[i]],
                 [exam_scores[i], good_pred[i]], 'g--', alpha=0.5)
axes[1].set_title(f"Best line — MSE = {good_mse:.1f}")
axes[1].set_xlabel("Hours Studied")
axes[1].set_ylabel("Exam Score")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The dashed lines show the errors — the distance between each point and the line. The good line (right) has much smaller errors than the bad line (left).

Training a model means: **find the w and b that minimize the loss function.**

In [ ]:
# Gradient descent — the iterative process
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 3))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3.2)
ax.axis("off")

steps = [
    (1.5,  "Random\nWeights", "#999"),
    (4.5,  "Calculate\nLoss", "#D9534F"),
    (7.5,  "Compute\nGradient", "#F0AD4E"),
    (10.5, "Update\nWeights", "#5CB85C"),
]

for x, label, color in steps:
    box = mpatches.FancyBboxPatch(
        (x - 1.1, 0.7), 2.2, 1.5, boxstyle="round,pad=0.15",
        facecolor=color, edgecolor="white", linewidth=2, alpha=0.85
    )
    ax.add_patch(box)
    ax.text(x, 1.45, label, ha="center", va="center",
            fontsize=11, fontweight="bold", color="white")

# Forward arrows
for i in range(len(steps) - 1):
    ax.annotate("", xy=(steps[i+1][0] - 1.15, 1.45),
                xytext=(steps[i][0] + 1.15, 1.45),
                arrowprops=dict(arrowstyle="-|>", color="#333", lw=2))

# Loop arrow from "Update Weights" back to "Calculate Loss"
ax.annotate("", xy=(steps[1][0] - 1.1, 2.5),
            xytext=(steps[3][0] + 1.1, 2.5),
            arrowprops=dict(arrowstyle="-|>", color="#4A90D9", lw=2.5,
                            connectionstyle="arc3,rad=-0.3"))
ax.text(7.5, 3.05, "Repeat until loss stops decreasing",
        ha="center", fontsize=10, fontstyle="italic",
        color="#4A90D9", fontweight="bold")

# Annotations below
ax.text(1.5, 0.25, "Start here", ha="center", fontsize=9, color="#999")
ax.text(7.5, 0.25, "\"Which way is downhill?\"",
        ha="center", fontsize=9, color="#F0AD4E", fontstyle="italic")

fig.suptitle("Gradient Descent: How Models Learn", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---

## Gradient Descent: How Models Learn

How does the model actually find the best line? It uses an algorithm called **gradient descent**.

Here's the intuition. Imagine you're standing on a hilly landscape in thick fog. You can't see the bottom of the valley, but you can feel the slope under your feet. What do you do? You take a step downhill. Then another. And another. Eventually you reach the lowest point.

That's gradient descent:

1. Start with random values for $w$ and $b$
2. Calculate the loss (how wrong are we?)
3. Figure out which direction to adjust $w$ and $b$ to reduce the loss (the **gradient** — the slope)
4. Take a small step in that direction
5. Repeat until the loss stops decreasing

The size of each step is called the **learning rate**. Too big and you overshoot. Too small and it takes forever. We'll explore this more in Module 3.

In [ ]:
# Let's visualize gradient descent
# We'll try different values of w (slope) and see how MSE changes

w_values = np.linspace(0, 16, 100)
mse_values = []

b_fixed = 25  # Fix b for simplicity, just vary w
for w in w_values:
    predictions = w * study_hours + b_fixed
    mse = np.mean((exam_scores - predictions) ** 2)
    mse_values.append(mse)

plt.figure(figsize=(8, 5))
plt.plot(w_values, mse_values, linewidth=2)
plt.xlabel("Weight (w)")
plt.ylabel("MSE (loss)")
plt.title("Loss Landscape: How MSE Changes with Different Weights")
plt.grid(True, alpha=0.3)

# Mark the minimum
best_w = w_values[np.argmin(mse_values)]
plt.axvline(x=best_w, color='red', linestyle='--', alpha=0.7, label=f"Best w ≈ {best_w:.1f}")
plt.legend()
plt.show()

print(f"The best weight is around {best_w:.1f}")
print(f"sklearn found: w = {model.coef_[0]:.1f}, b = {model.intercept_:.1f}")

The curve has a clear minimum — a single best value of $w$. Gradient descent starts somewhere on this curve and slides downhill to the bottom.

For linear regression, this curve is always a smooth bowl shape (convex), so gradient descent is guaranteed to find the best answer. For more complex models (neural networks), the landscape is bumpier — but the idea is the same.

---

## Training a Linear Regression with scikit-learn

In practice, you don't code gradient descent by hand. scikit-learn does it for you in two lines.

In [ ]:
# Step 1: Prepare the data
X = study_hours.reshape(-1, 1)  # sklearn expects a 2D array for features
y = exam_scores

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Step 2: Create the model and train it
model = LinearRegression()
model.fit(X_train, y_train)  # This is where learning happens

print(f"Weight (slope): {model.coef_[0]:.2f}")
print(f"Bias (intercept): {model.intercept_:.2f}")
print(f"\nEquation: score = {model.coef_[0]:.2f} × hours + {model.intercept_:.2f}")

The model learned: each hour of studying is worth about 8 more points on the exam, with a base score around 25.

That's meaningful — you can interpret linear regression. This interpretability is one of its strengths.

In [ ]:
# Step 3: Make predictions on the test set
y_pred = model.predict(X_test)

# Compare predictions to actual values
for actual, predicted in zip(y_test, y_pred):
    print(f"Actual: {actual:.1f}  Predicted: {predicted:.1f}  Error: {actual - predicted:+.1f}")

In [ ]:
# Step 4: Evaluate with MSE
from sklearn.metrics import mean_squared_error

train_mse = mean_squared_error(y_train, model.predict(X_train))
test_mse = mean_squared_error(y_test, y_pred)

print(f"Training MSE: {train_mse:.2f}")
print(f"Test MSE:     {test_mse:.2f}")
print(f"\nTest RMSE:    {np.sqrt(test_mse):.2f} (in same units as scores)")

**RMSE** (Root Mean Squared Error) is just the square root of MSE. It's useful because it's in the same units as your target — here, exam score points. An RMSE of ~5 means the model's predictions are off by about 5 points on average.

In [ ]:
# Visualize the fit
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, label="Training data", s=80, zorder=5)
plt.scatter(X_test, y_test, label="Test data", s=80, marker="s", zorder=5)

line_x = np.linspace(0, 11, 100).reshape(-1, 1)
line_y = model.predict(line_x)
plt.plot(line_x, line_y, 'r-', linewidth=2, label="Model")

plt.xlabel("Hours Studied")
plt.ylabel("Exam Score")
plt.title("Linear Regression: Study Hours → Exam Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

## Multiple Features

So far we used one feature. But most real problems have many features. Linear regression handles this naturally:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b$$

One weight per feature. The model figures out how important each feature is.

In [ ]:
# Let's use the California housing dataset from last session
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

train_mse = mean_squared_error(y_train, model.predict(X_train))
test_mse = mean_squared_error(y_test, model.predict(X_test))

print(f"Training MSE:  {train_mse:.4f}")
print(f"Test MSE:      {test_mse:.4f}")
print(f"Test RMSE:     {np.sqrt(test_mse):.4f} (in $100K)")
print(f"\nThat's roughly ${np.sqrt(test_mse) * 100000:,.0f} average error")

In [ ]:
# Which features matter most?
feature_importance = pd.Series(model.coef_, index=housing.feature_names)
print("Feature weights:")
print(feature_importance.to_string())

The weights tell you how each feature affects the prediction. A large positive weight means that feature increases the predicted price. A negative weight means it decreases it.

Careful though — comparing weights directly only works well when features are on the same scale. Latitude and median income are measured in very different units. We'll learn about feature scaling later.

---

## The sklearn Pattern

Every model in scikit-learn follows the same pattern:

```python
from sklearn.some_module import SomeModel

model = SomeModel()           # 1. Create the model
model.fit(X_train, y_train)   # 2. Train it
predictions = model.predict(X_test)  # 3. Predict
```

This is the same whether you're using linear regression, decision trees, or support vector machines. Learn this pattern once and you can use any sklearn model.

---

## Exercises

### Exercise 1: Interpret the model

Using the study hours model we trained earlier (score = w × hours + b):
- What score does the model predict for someone who studies 5.5 hours?
- What score does it predict for 0 hours? Does this make sense?
- According to the model, how many extra points do you get for each additional hour of studying?

In [ ]:
# Exercise 1: Your code here


### Exercise 2: Loss calculation

A model predicts: [50, 60, 70, 80]. The actual values are: [48, 65, 68, 82].

Calculate the MSE by hand (you can use numpy). Then calculate the RMSE.

In [ ]:
# Exercise 2: Your code here
predicted = np.array([50, 60, 70, 80])
actual = np.array([48, 65, 68, 82])


### Exercise 3: Your own regression

Create a small dataset (at least 8 points) of something that has a linear relationship — for example, temperature vs ice cream sales, or age vs height for kids. Train a linear regression and plot the result.

Steps:
1. Create X and y arrays
2. Split into train/test
3. Train a LinearRegression
4. Print the weight and bias
5. Plot the data and the fitted line

In [ ]:
# Exercise 3: Your code here


---

## Summary

**Linear regression:**
- Finds the best straight line through data: $\hat{y} = w \cdot x + b$
- "Best" = lowest loss (smallest errors between predictions and actual values)
- With multiple features: one weight per feature

**Loss functions:**
- MSE (Mean Squared Error) measures average squared prediction error
- RMSE is the square root of MSE — same units as your target
- Lower loss = better model

**Gradient descent (the intuition):**
- Start with random weights
- Calculate loss, figure out which way is "downhill," take a step
- Repeat until loss stops decreasing
- Learning rate controls the step size

**The sklearn pattern:** `create → fit → predict` — works for every model

---

**Next session (Lab):** You'll explore a real dataset hands-on — load it, inspect it, visualize it with pandas and matplotlib.